In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import json
import os

# Load your best trained CNN
model = keras.models.load_model('../models/cnn_best.keras')

# Load normalisation parameters
norm_mean = np.load('../models/norm_mean.npy')
norm_std  = np.load('../models/norm_std.npy')

# Load config
with open('../models/model_config.json') as f:
    config = json.load(f)

print("Model loaded successfully")
print(f"Input shape:  {model.input_shape}")
print(f"Output shape: {model.output_shape}")
print(f"Production threshold: {config['threshold']}")

Model loaded successfully
Input shape:  (None, 100, 6)
Output shape: (None, 1)
Production threshold: 0.35


In [2]:
# Load your training windows for calibration
# Quantisation needs real data samples to measure
# the range of values flowing through the model
X_train = np.load('../data/processed/X_train_raw.npy')

# Normalise using saved parameters
X_train_norm = (X_train - norm_mean) / norm_std
X_train_norm = X_train_norm.astype(np.float32)

# Use 200 random samples for calibration
calibration_indices = np.random.choice(
    len(X_train_norm), 200, replace=False
)
calibration_data = X_train_norm[calibration_indices]

print(f"Calibration data shape: {calibration_data.shape}")
print(f"Value range: {calibration_data.min():.2f} to {calibration_data.max():.2f}")

Calibration data shape: (200, 100, 6)
Value range: -41.55 to 41.62


In [3]:
def representative_dataset():
    """
    Feeds real data samples to the converter so it can
    measure the range of activations and quantise accurately.
    """
    for i in range(len(calibration_data)):
        sample = calibration_data[i:i+1]  # shape: (1, 100, 6)
        yield [sample]


# Set up the converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable quantisation
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

# Keep input and output as float32 so Flutter can
# feed raw float arrays without extra conversion
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

print("Converting model to TFLite...")
tflite_model = converter.convert()
print("Conversion complete!")

# Save the .tflite file
os.makedirs('../models', exist_ok=True)
tflite_path = '../models/crash_detector.tflite'

with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# Check file size
size_kb = len(tflite_model) / 1024
size_mb = size_kb / 1024

print(f"\nFile saved: {tflite_path}")
print(f"File size:  {size_kb:.1f} KB  ({size_mb:.3f} MB)")
print(f"Size check: {'PASS' if size_mb < 5.0 else 'FAIL'}  (must be < 5MB)")

Converting model to TFLite...
INFO:tensorflow:Assets written to: C:\Users\sandu\AppData\Local\Temp\tmp8eoi5gap\assets


INFO:tensorflow:Assets written to: C:\Users\sandu\AppData\Local\Temp\tmp8eoi5gap\assets


Saved artifact at 'C:\Users\sandu\AppData\Local\Temp\tmp8eoi5gap'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 6), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1860209436688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209438608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209438800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209435728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209437648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209436880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209440336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209441104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209441488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1860209442448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  18602094422

D:\My Projects\rideaway_ml\venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Conversion complete!

File saved: ../models/crash_detector.tflite
File size:  59.9 KB  (0.059 MB)
Size check: PASS  (must be < 5MB)


In [4]:
# Load test data
X_test      = np.load('../data/processed/X_test_raw.npy')
y_test      = np.load('../data/processed/y_test.npy')
X_test_norm = ((X_test - norm_mean) / norm_std).astype(np.float32)

# Run inference using the TFLite interpreter
# (exactly how your Flutter app will run it)
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input tensor shape:  {input_details[0]['shape']}")
print(f"Output tensor shape: {output_details[0]['shape']}")
print(f"Input dtype:         {input_details[0]['dtype']}")

# Run every test window through the TFLite interpreter
tflite_scores = []

for i in range(len(X_test_norm)):
    # Reshape to (1, 100, 6) — one window at a time
    input_data = X_test_norm[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    tflite_scores.append(output[0][0])

tflite_scores = np.array(tflite_scores)
print(f"\nInference complete — {len(tflite_scores)} windows processed")

Input tensor shape:  [  1 100   6]
Output tensor shape: [1 1]
Input dtype:         <class 'numpy.float32'>

Inference complete — 540 windows processed


D:\My Projects\rideaway_ml\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [5]:
from sklearn.metrics import confusion_matrix, f1_score

threshold = config['threshold']  # 0.35

# Original CNN predictions
keras_scores = model.predict(X_test_norm, verbose=0).flatten()
keras_pred   = (keras_scores >= threshold).astype(int)

# TFLite predictions
tflite_pred  = (tflite_scores >= threshold).astype(int)

# Original CNN metrics
tn, fp, fn, tp = confusion_matrix(y_test, keras_pred).ravel()
keras_recall = tp / (tp + fn)
keras_fpr    = fp / (fp + tn)

# TFLite metrics
tn, fp, fn, tp = confusion_matrix(y_test, tflite_pred).ravel()
tflite_recall = tp / (tp + fn)
tflite_fpr    = fp / (fp + tn)
tflite_f1     = f1_score(y_test, tflite_pred)

print("=== TFLite Verification ===")
print(f"{'Metric':<25} {'Original CNN':<16} {'TFLite':<16} {'Match?'}")
print("-" * 65)
print(f"{'Recall':<25} {keras_recall:<16.3f} {tflite_recall:<16.3f} {'OK' if abs(keras_recall-tflite_recall) < 0.01 else 'DIFF'}")
print(f"{'False positive rate':<25} {keras_fpr:<16.3f} {tflite_fpr:<16.3f} {'OK' if abs(keras_fpr-tflite_fpr) < 0.01 else 'DIFF'}")
print(f"\nTP: {tp}  FP: {fp}  TN: {tn}  FN: {fn}")
print(f"F1 score: {tflite_f1:.3f}")

# Agreement rate between original and TFLite
agreement = (keras_pred == tflite_pred).mean()
print(f"\nPrediction agreement: {agreement*100:.1f}%  (want > 99%)")

=== TFLite Verification ===
Metric                    Original CNN     TFLite           Match?
-----------------------------------------------------------------
Recall                    0.969            0.957            DIFF
False positive rate       0.018            0.018            OK

TP: 154  FP: 7  TN: 372  FN: 7
F1 score: 0.957

Prediction agreement: 99.3%  (want > 99%)


In [6]:
import time

# Simulate 100 windows and time how long it takes
n_timing_runs = 100
test_window   = X_test_norm[0:1].astype(np.float32)

start = time.perf_counter()
for _ in range(n_timing_runs):
    interpreter.set_tensor(input_details[0]['index'], test_window)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]['index'])
end = time.perf_counter()

avg_ms = ((end - start) / n_timing_runs) * 1000

print(f"Average inference time: {avg_ms:.2f} ms per window")
print(f"Latency check: {'PASS' if avg_ms < 100 else 'NOTE: may be slower on min-spec device'}")
print(f"\nNote: this is on your PC. On a mid-range Android phone")
print(f"expect 2-5x slower, but still well under 100ms target.")

Average inference time: 0.08 ms per window
Latency check: PASS

Note: this is on your PC. On a mid-range Android phone
expect 2-5x slower, but still well under 100ms target.


In [7]:
print("=== Files ready for Flutter integration ===")
print()

files = {
    '../models/crash_detector.tflite': 'Main model file — copy to Flutter assets/',
    '../models/model_config.json':     'Threshold and config — copy to Flutter assets/',
    '../models/norm_mean.npy':         'Normalisation mean — embed values in Flutter code',
    '../models/norm_std.npy':          'Normalisation std  — embed values in Flutter code',
}

for path, description in files.items():
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if exists else 0
    status = "EXISTS" if exists else "MISSING"
    print(f"  {status}  {path}")
    print(f"          {description}")
    print(f"          Size: {size:.1f} KB")
    print()

# Print the normalisation values Flutter will need
print("=== Normalisation values for Flutter ===")
print("Copy these into your Flutter Dart code:\n")
print("final normMean = [")
for i, v in enumerate(norm_mean):
    axis = ['acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z'][i]
    print(f"  {v:.6f},  // {axis}")
print("];\n")
print("final normStd = [")
for i, v in enumerate(norm_std):
    axis = ['acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z'][i]
    print(f"  {v:.6f},  // {axis}")
print("];")

=== Files ready for Flutter integration ===

  EXISTS  ../models/crash_detector.tflite
          Main model file — copy to Flutter assets/
          Size: 59.9 KB

  EXISTS  ../models/model_config.json
          Threshold and config — copy to Flutter assets/
          Size: 0.3 KB

  EXISTS  ../models/norm_mean.npy
          Normalisation mean — embed values in Flutter code
          Size: 0.2 KB

  EXISTS  ../models/norm_std.npy
          Normalisation std  — embed values in Flutter code
          Size: 0.2 KB

=== Normalisation values for Flutter ===
Copy these into your Flutter Dart code:

final normMean = [
  -0.025634,  // acc_x
  -0.059088,  // acc_y
  -0.002480,  // acc_z
  0.021144,  // gyro_x
  0.021613,  // gyro_y
  0.029625,  // gyro_z
];

final normStd = [
  2.461121,  // acc_x
  2.511723,  // acc_y
  2.507216,  // acc_z
  0.943263,  // gyro_x
  1.049961,  // gyro_y
  1.076128,  // gyro_z
];
